# MCTS + SAC vs. raw SAC — local experiment harness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/re_org/notebooks/mcts_experiments.ipynb)

Compares an MCTS-guided agent (using `RL models/soft_actor_critic.keras`
for the policy prior and value estimate) against the same SAC model
running raw greedy argmax, so you can quickly measure whether MCTS at
inference improves on the policy alone — and which configuration of MCTS
gives the biggest lift.

**Local or Colab.** On Colab, Cell 1 clones the repo into `/content/`
and uses the SAC model from `RL models/soft_actor_critic.keras`. Locally,
it walks up to the repo root from the notebook's working directory.
A single match of 20 games at 50 simulations/move takes roughly 1–2
minutes on CPU, ~20–30 seconds on a Colab T4 GPU.

The agent code lives in `src/mcts.py`. Edit the cells below to vary
`n_simulations`, `c_puct`, and `value_method`; the sweep cell at the
bottom does this for you.


In [ ]:
# Cell 1 — environment setup + load the SAC model.
# Detects Colab vs local; on Colab, clones the repo. Locally, walks up
# to the repo root from the notebook's CWD.
import os
import sys
import subprocess
from pathlib import Path

GITHUB_OWNER  = 'Stiles-Clements1'
GITHUB_REPO   = 'connect4-rl-arena'
GITHUB_BRANCH = 're_org'
GITHUB_URL    = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_ROOT = Path(f'/content/{GITHUB_REPO}')
    if not REPO_ROOT.exists():
        print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}')
        subprocess.run(
            ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        print(f'Repo already at {REPO_ROOT}; pulling latest {GITHUB_BRANCH}.')
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch',  '--quiet', 'origin'], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout','--quiet', GITHUB_BRANCH], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull',   '--quiet', 'origin', GITHUB_BRANCH], check=False)
    subprocess.run(['pip', 'install', '-q', 'tqdm', 'pandas'], check=False)
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / 'src' / 'game_engine.py').exists()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find repo root from {here}. '
            f'Run the notebook from inside a clone of connect4-rl-arena.'
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Purge cached src.* imports so a freshly-pulled session picks up new code.
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

print(f'Running in {"Colab" if IS_COLAB else "local"} mode.')
print(f'REPO_ROOT = {REPO_ROOT}')

# Quiet TF down BEFORE we import anything that loads it.
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass
HARDWARE = f'GPU ({_gpus[0].name.split(":")[-1]})' if _gpus else 'CPU only'
print(f'Hardware  = {HARDWARE}')

import numpy as np
from src import model_loader as ml
from src.eval import ModelAgent, play_match, format_result
from src.mcts import MCTSAgent

# Load the SAC model directly. Encoding is auto-detected from input shape:
#   (None, 6, 7, 2) -> 'B'   (None, 6, 7, 1) -> 'A'   (None, 42, 2) -> 'B_flat'
SAC_PATH = REPO_ROOT / 'RL models' / 'soft_actor_critic.keras'
keras_model = tf.keras.models.load_model(str(SAC_PATH), compile=False)

# Get the input shape robustly across Keras versions. `model.input` can be
# either a single KerasTensor (older Keras) or a list of them (Keras 3 /
# multi-input models); `model.inputs` (plural) is always a list, so use
# the first element.
try:
    in_shape = tuple(keras_model.input.shape)
except (AttributeError, TypeError):
    in_shape = tuple(keras_model.inputs[0].shape)

trailing = in_shape[-3:] if len(in_shape) >= 3 else None
if trailing == (6, 7, 2):
    encoding = 'B'
elif trailing == (6, 7, 1):
    encoding = 'A'
elif in_shape[-2:] == (42, 2):
    encoding = 'B_flat'
else:
    raise ValueError(f'Unrecognised input shape {in_shape}')

sac_wrapper = ml.ModelWrapper(keras_model, encoding, name='SAC')
print(f'\nLoaded SAC: input shape {in_shape}, encoding={encoding}')

# Raw SAC opponent — greedy argmax over the policy + tactical overrides ON
# (matching the configuration that was submitted to the tournament).
sac_raw = ModelAgent(sac_wrapper, name='sac_raw', greedy=True, use_tactics=True)
print(f'Ready. SAC raw agent: {sac_raw.name}')


In [ ]:
# Cell 2 — helper: build an MCTS agent with a given config and play it
# vs the raw SAC agent for n_games. Returns the eval.MatchResult and prints
# a formatted summary.
def run_mcts_match(
    n_simulations=100,
    c_puct=1.4,
    value_method='mean_q',
    use_tactics=True,
    n_games=20,
    random_init_moves=4,
    add_root_noise=False,
):
    mcts_agent = MCTSAgent(
        sac_wrapper,
        n_simulations=n_simulations,
        c_puct=c_puct,
        value_method=value_method,
        use_tactics=use_tactics,
        add_root_noise=add_root_noise,
    )
    print(f"Match: {mcts_agent.name}  vs  {sac_raw.name}")
    print(f"  n_games={n_games}, c_puct={c_puct}, "
          f"random_init={random_init_moves}, tactics={use_tactics}")
    result = play_match(
        mcts_agent, sac_raw,
        n_games=n_games,
        random_init_moves=random_init_moves,
        progress=True,
    )
    print(format_result(result))
    print()
    return result


In [ ]:
# Cell 3 — quick single-config check (~1–2 minutes on CPU).
# Verifies the wiring works and gives a first directional reading.
result = run_mcts_match(
    n_simulations=50,
    c_puct=1.4,
    value_method='mean_q',
    n_games=20,
)

# Interpretation:
#   - a_win_rate >> 50%  -> MCTS adds real value over raw SAC.
#   - a_win_rate ~ 50%   -> MCTS roughly equivalent at this n_simulations.
#   - a_win_rate < 50%   -> MCTS is hurting (likely n_simulations too low,
#                          or value_method/c_puct mismatched to the model).


In [ ]:
# Cell 4 — sweep configurations and rank them.
# This cell takes longer (~10–20 min on CPU). Reduce n_games or the
# config list if you want a faster pass.
import pandas as pd

CONFIGS = [
    dict(n_simulations=50,  c_puct=1.4, value_method='mean_q'),
    dict(n_simulations=100, c_puct=1.4, value_method='mean_q'),
    dict(n_simulations=200, c_puct=1.4, value_method='mean_q'),
    dict(n_simulations=100, c_puct=1.0, value_method='mean_q'),
    dict(n_simulations=100, c_puct=2.0, value_method='mean_q'),
    dict(n_simulations=100, c_puct=1.4, value_method='rollout'),
    dict(n_simulations=200, c_puct=1.4, value_method='rollout'),
]
N_GAMES_PER_CONFIG = 30  # bump to 50–100 if you want tighter CIs

rows = []
for cfg in CONFIGS:
    res = run_mcts_match(n_games=N_GAMES_PER_CONFIG, **cfg)
    rows.append({
        'n_sims':       cfg['n_simulations'],
        'c_puct':       cfg['c_puct'],
        'value':        cfg['value_method'],
        'mcts_wins':    res.a_wins,
        'sac_wins':     res.b_wins,
        'draws':        res.draws,
        'mcts_winrate': res.a_win_rate,
        'avg_len':      round(res.avg_length, 1),
    })

df = pd.DataFrame(rows).sort_values('mcts_winrate', ascending=False)
print('\n=== Sweep summary (sorted by MCTS win rate) ===')
print(df.to_string(index=False))


## Reading the results

- **Win rate above ~55%**: MCTS+SAC is meaningfully stronger than raw SAC
  at that configuration. The bigger the margin, the better-justified the
  AlphaZero recommendation in the report (your tournament loss was to
  agents using exactly this kind of inference-time search).
- **Win rate near 50%**: at this `n_simulations` budget the search isn't
  adding signal beyond the policy. Try doubling `n_simulations` or
  switching `value_method`.
- **Win rate below 50%**: MCTS is actively hurting. Most likely causes,
  in order: (1) `n_simulations` too low so PUCT explores noisy children
  uniformly; (2) `value_method='mean_q'` but the SAC Q heads are poorly
  calibrated for inference (try `'rollout'`); (3) `c_puct` too small
  (try 2.0).

## Things to try next

- **Add an MCTS submission.**  If a configuration consistently wins
  ≥65% of games against raw SAC, that's the version to submit in any
  rematch. The same `MCTSAgent` class plugs into the live-match
  notebook via the `select_move(board, player)` interface.
- **Vary `n_simulations` higher** if you have GPU. 400 simulations/move
  is roughly the strength regime AlphaZero used at "casual" play; 1600+
  is tournament strength.
- **Disable `use_tactics`** to see the *pure* MCTS vs *pure* policy
  contrast. With tactics on, the immediate-win/block layer fires before
  MCTS in many positions, so part of the agent's strength comes from
  the heuristic rather than the search.


In [ ]:
# Cell 6 — Visualise the sweep.
# Run this AFTER Cell 4 (it expects the `df` that the sweep produced).
import matplotlib.pyplot as plt

# Sort weakest -> strongest so the best config appears at the top of the chart
df_plot = df.sort_values('mcts_winrate', ascending=True).reset_index(drop=True)

# Per-row config label for the y-axis
labels = [
    f"n={int(r['n_sims']):>3}, c={r['c_puct']:.1f}, {r['value']}"
    for _, r in df_plot.iterrows()
]
winrates_pct = df_plot['mcts_winrate'].to_numpy() * 100

# Green if MCTS wins, red if it loses, grey at parity
colors = [
    '#2ecc71' if w > 50 else '#e74c3c' if w < 50 else '#999'
    for w in winrates_pct
]

fig, ax = plt.subplots(figsize=(10, max(3.5, 0.55 * len(df_plot))))
bars = ax.barh(
    labels, winrates_pct, color=colors, edgecolor='black', linewidth=0.4,
)
ax.axvline(50, color='black', linestyle='--', linewidth=1, alpha=0.7,
           label='50% (parity)')

# Annotate each bar: win-rate %, then W-L-D counts
for bar, (_, row) in zip(bars, df_plot.iterrows()):
    w = row['mcts_winrate'] * 100
    ax.text(
        max(w, 1) + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{w:.0f}%  ({int(row['mcts_wins'])}-{int(row['sac_wins'])}-{int(row['draws'])})",
        va='center', ha='left', fontsize=9,
    )

ax.set_xlabel('MCTS+SAC win rate vs. raw SAC (%)')
ax.set_xlim(0, 105)
ax.set_title('MCTS configuration sweep — win rate against raw SAC\n'
             '(annotation = MCTS-wins–SAC-wins–draws)')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Print the strongest config and a one-line verdict
best  = df.sort_values('mcts_winrate', ascending=False).iloc[0]
worst = df.sort_values('mcts_winrate', ascending=True).iloc[0]
print(f"\nStrongest: n_sims={int(best['n_sims'])}, c_puct={best['c_puct']}, "
      f"value_method={best['value']}  ->  {best['mcts_winrate']*100:.0f}% win rate")
print(f"Weakest  : n_sims={int(worst['n_sims'])}, c_puct={worst['c_puct']}, "
      f"value_method={worst['value']}  ->  {worst['mcts_winrate']*100:.0f}% win rate")
if best['mcts_winrate'] > 0.55:
    print(f"\nVerdict: MCTS adds value at this configuration. "
          f"That config is what to wire into a tournament submission.")
elif best['mcts_winrate'] >= 0.50:
    print(f"\nVerdict: MCTS is roughly tying raw SAC at the best config. "
          f"Try doubling n_simulations or adding higher c_puct values to the sweep.")
else:
    print(f"\nVerdict: MCTS underperforms raw SAC across this sweep. "
          f"Worth trying value_method='rollout' if not already in the sweep, "
          f"or doubling n_simulations.")
